# Impor Libraries

In [95]:
# ============================================
# CELL 1: Import Libraries
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler  # ⬅️ TAMBAHKAN untuk scaling
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
import joblib  # ⬅️ TAMBAHKAN untuk save/load model

print("✅ Semua library berhasil diimport!")

✅ Semua library berhasil diimport!


# Cell 2: Load Data


In [96]:
# ============================================
# CELL 2: Load Data & Preprocessing
# ============================================

# 1. Load Dataset Tabular
df = pd.read_csv('dataset_stunting_clean.csv')

# Tampilkan info dataset
print("Shape dataset:", df.shape)
print("\n5 Baris Pertama Dataset:")
print(df.head())

# 2. Feature Engineering & Encoding
# Encode jenis_kelamin
df['jenis_kelamin_encoded'] = df['jenis_kelamin'].apply(lambda x: 0 if x == 'Laki-laki' else 1)

# Hitung IMT (Berat Badan / (Tinggi Badan/100)^2)
df['imt'] = df['berat_badan'] / ((df['tinggi_badan'] / 100) ** 2)

# Hitung Tinggi per Usia (menggunakan formula sederhana, bisa disesuaikan)
df['tinggi_per_usia'] = df['tinggi_badan'] / (df['usia_bulan'] * 1.5 + 45)

# Encode target variable (assuming 'status_tb_u' is the target)
df['target'] = df['status_tb_u'].apply(lambda x: 1 if x == 'Stunting' else 0)


# 3. Definisikan fitur dan target
feature_columns = ['usia_bulan', 'jenis_kelamin_encoded', 'tinggi_badan',
                   'berat_badan', 'imt', 'tinggi_per_usia']

X = df[feature_columns].values
y = df['target'].values # Use the newly created 'target' column

print(f"\nFitur yang digunakan: {feature_columns}")
print(f"Jumlah sampel: {len(X)}")
print(f"Distribusi target:\n{pd.Series(y).value_counts()}") # Changed to use value_counts for better distribution view

# 4. Split data (80% train, 20% test)
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nSplit data:")
print(f"X_train_raw: {X_train_raw.shape}")
print(f"X_test: {X_test.shape}")

# 5. Inisialisasi dan FIT Scaler (hanya dari data train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)

print(f"\nScaling selesai!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"Mean setelah scaling: {X_train_scaled.mean():.10f}")
print(f"Std setelah scaling: {X_train_scaled.std():.10f}")

# 6. SMOTE untuk menyeimbangkan kelas
print("\nSebelum SMOTE:")
print(f"Jumlah kelas 0: {sum(y_train_raw == 0)}")
print(f"Jumlah kelas 1: {sum(y_train_raw == 1)}")

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_scaled, y_train_raw)

print(f"\nSetelah SMOTE:")
print(f"Jumlah kelas 0: {sum(y_train == 0)}")
print(f"Jumlah kelas 1: {sum(y_train == 1)}")

# 7. Scaling untuk data test (gunakan scaler yang sudah fit)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Data siap untuk training!")
print(f"X_train shape: {X_train.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

Shape dataset: (40066, 11)

5 Baris Pertama Dataset:
   id_anak jenis_kelamin  usia_bulan  berat_badan  tinggi_badan status_bb_u  \
0        1     Perempuan        54.0         13.2          97.5      Normal   
1        2     Laki-laki        44.0         12.0          92.0      Normal   
2        3     Laki-laki        57.0         14.0          97.0      Normal   
3        4     Laki-laki        26.0         11.0          79.0      Normal   
4        5     Perempuan        59.0         14.6          98.0      Normal   

   zscore_bb_u status_tb_u  zscore_tb_u status_bb_tb  zscore_bb_tb  
0        -1.94    Stunting        -2.11       Normal         -0.95  
1        -1.92    Stunting        -2.22       Normal         -0.88  
2        -1.90    Stunting        -2.58       Normal         -0.48  
3        -1.15    Stunting        -3.11       Normal          0.68  
4        -1.66    Stunting        -2.49       Normal         -0.18  

Fitur yang digunakan: ['usia_bulan', 'jenis_kelamin_encod

# Cell 3: Pelatihan Model Random Forest


In [97]:
# ============================================
# CELL 3: Training Model Random Forest
# ============================================

# Inisialisasi model dengan parameter optimal
model_sidias = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)

print("Model RandomForestClassifier dibuat dengan parameter:")
print(f"- n_estimators: 200")
print(f"- max_depth: 12")
print(f"- class_weight: 'balanced'")

# Pelatihan model
print("\nMemulai training...")
model_sidias.fit(X_train, y_train)
print("✅ Training selesai!")

# Simpan model dan scaler untuk keperluan deployment
joblib.dump(model_sidias, 'model_sidias_rf.joblib')
joblib.dump(scaler, 'scaler_sidias.joblib')
print("\n✅ Model disimpan sebagai 'model_sidias_rf.joblib'")
print("✅ Scaler disimpan sebagai 'scaler_sidias.joblib'")

Model RandomForestClassifier dibuat dengan parameter:
- n_estimators: 200
- max_depth: 12
- class_weight: 'balanced'

Memulai training...
✅ Training selesai!

✅ Model disimpan sebagai 'model_sidias_rf.joblib'
✅ Scaler disimpan sebagai 'scaler_sidias.joblib'


# Cell 4: Evaluasi Metrik Baru

In [98]:
# ============================================
# CELL 4: Evaluasi Model
# ============================================

# 1. Prediksi pada Data Uji (gunakan X_test_scaled)
y_pred = model_sidias.predict(X_test_scaled)

# 2. Perhitungan Metrik Evaluasi
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
prec = precision_score(y_test, y_pred, average='weighted')
rec = recall_score(y_test, y_pred, average='weighted')

print("="*50)
print("     EVALUASI AKHIR MODEL SIDIAS")
print("="*50)
print(f"Akurasi     : {acc*100:.2f}% (Target >= 85%)")
print(f"F1-Score    : {f1:.4f}")
print(f"Precision   : {prec:.4f}")
print(f"Recall      : {rec:.4f}")
print("="*50)

# 3. Laporan Klasifikasi Detail
print("\n📊 Laporan Detail Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Stunting']))

# 4. Verifikasi Kelulusan KPI
if acc >= 0.85:
    print("\n✅ STATUS: Model LULUS KPI Akurasi dan Siap Digunakan.")
else:
    print("\n❌ STATUS: Akurasi di bawah target 85%. Perlu optimasi lebih lanjut.")

     EVALUASI AKHIR MODEL SIDIAS
Akurasi     : 95.74% (Target >= 85%)
F1-Score    : 0.9575
Precision   : 0.9577
Recall      : 0.9574

📊 Laporan Detail Klasifikasi:
              precision    recall  f1-score   support

      Normal       0.97      0.96      0.96      4774
    Stunting       0.94      0.96      0.95      3240

    accuracy                           0.96      8014
   macro avg       0.95      0.96      0.96      8014
weighted avg       0.96      0.96      0.96      8014


✅ STATUS: Model LULUS KPI Akurasi dan Siap Digunakan.


In [99]:
# ============================================
# CELL 5: Load Model yang Tersimpan
# ============================================

# Memuat model dan scaler yang telah disimpan sebelumnya
try:
    model_sidias = joblib.load('model_sidias_rf.joblib')
    scaler = joblib.load('scaler_sidias.joblib')
    print("✅ Model berhasil dimuat dari 'model_sidias_rf.joblib'")
    print("✅ Scaler berhasil dimuat dari 'scaler_sidias.joblib'")
    print("\nModel siap melakukan inferensi!")
except FileNotFoundError as e:
    print(f"❌ ERROR: File tidak ditemukan - {e}")
    print("Pastikan Anda telah menjalankan Cell 3 (training) terlebih dahulu.")

✅ Model berhasil dimuat dari 'model_sidias_rf.joblib'
✅ Scaler berhasil dimuat dari 'scaler_sidias.joblib'

Model siap melakukan inferensi!


# Fungsi Pemrosesan Data Input (Inference Engine)

In [100]:
# ============================================
# CELL 6: Fungsi Prediksi (Inference Engine)
# ============================================

def predict_stunting(usia_bulan, jenis_kelamin, tinggi_badan, berat_badan):
    """
    Fungsi untuk melakukan inferensi model SIDIAS dengan input REALISTIS.

    PARAMETER (dalam bentuk asli, bukan scaled):
    -------------------------------------------------
    usia_bulan    : float - usia dalam bulan (contoh: 36.0)
    jenis_kelamin : str   - "Laki-laki" atau "Perempuan"
    tinggi_badan  : float - tinggi dalam cm (contoh: 92.0)
    berat_badan   : float - berat dalam kg (contoh: 12.0)

    RETURNS:
    -------------------------------------------------
    status       : str    - "Stunting" atau "Normal"
    confidence   : float  - persentase keyakinan (0-100)
    probabilities: dict   - probabilitas untuk setiap kelas
    """

    # Langkah 1: Hitung fitur turunan (IMT)
    tinggi_m = tinggi_badan / 100  # konversi cm ke meter
    imt = berat_badan / (tinggi_m ** 2)

    # Langkah 2: Hitung fitur turunan (Tinggi per Usia)
    # Formula sederhana: tinggi_aktual / (usia * 1.5 + 45)
    # (Sesuaikan dengan referensi WHO jika perlu)
    tinggi_per_usia = tinggi_badan / (usia_bulan * 1.5 + 45)

    # Langkah 3: Encoding jenis kelamin
    if jenis_kelamin.lower() == "laki-laki":
        gender_encoded = 0
    elif jenis_kelamin.lower() == "perempuan":
        gender_encoded = 1
    else:
        raise ValueError("Jenis kelamin harus 'Laki-laki' atau 'Perempuan'")

    # Langkah 4: Buat array input sesuai urutan fitur
    input_features = np.array([[
        usia_bulan,
        gender_encoded,
        tinggi_badan,
        berat_badan,
        imt,
        tinggi_per_usia
    ]])

    print(f"\n📝 Input features:")
    print(f"   - Usia: {usia_bulan} bulan")
    print(f"   - Jenis Kelamin: {jenis_kelamin} → encoded: {gender_encoded}")
    print(f"   - Tinggi: {tinggi_badan} cm")
    print(f"   - Berat: {berat_badan} kg")
    print(f"   - IMT: {imt:.2f}")
    print(f"   - Tinggi/Usia: {tinggi_per_usia:.4f}")

    # Langkah 5: Scaling input menggunakan scaler yang sudah disimpan
    input_scaled = scaler.transform(input_features)

    # Langkah 6: Prediksi
    prediction = model_sidias.predict(input_scaled)
    probabilities = model_sidias.predict_proba(input_scaled)

    # Langkah 7: Interpretasi hasil
    status = "Stunting" if prediction[0] == 1 else "Normal"
    confidence = probabilities[0][prediction[0]] * 100

    return status, confidence, {
        "Normal": probabilities[0][0],
        "Stunting": probabilities[0][1]
    }

print("✅ Fungsi predict_stunting() siap digunakan!")

✅ Fungsi predict_stunting() siap digunakan!


# Simulasi Inferensi (Input Manual dari Kader)

In [101]:
# ============================================
# CELL 7: Simulasi Inferensi (Input Realistis)
# ============================================

# Contoh 1: Balita dengan potensi stunting
print("="*50)
print("        SIMULASI INFERENSIS SIDIAS")
print("="*50)

# Data balita (usia 36 bulan / 3 tahun, Laki-laki, tinggi 92 cm, berat 12 kg)
usia = 2.0
gender = "Perempuan"
tinggi = 70.6
berat = 70.0

# Eksekusi Inferensi
status, confidence, probs = predict_stunting(usia, gender, tinggi, berat)

print("\n" + "="*50)
print("              HASIL DIAGNOSIS")
print("="*50)
print(f"Status Gizi Balita : {status}")
print(f"Tingkat Kepercayaan: {confidence:.2f}%")
print(f"\nProbabilitas:")
print(f"   - Normal   : {probs['Normal']*100:.2f}%")
print(f"   - Stunting : {probs['Stunting']*100:.2f}%")
print("="*50)

        SIMULASI INFERENSIS SIDIAS

📝 Input features:
   - Usia: 2.0 bulan
   - Jenis Kelamin: Perempuan → encoded: 1
   - Tinggi: 70.6 cm
   - Berat: 70.0 kg
   - IMT: 140.44
   - Tinggi/Usia: 1.4708

              HASIL DIAGNOSIS
Status Gizi Balita : Normal
Tingkat Kepercayaan: 83.14%

Probabilitas:
   - Normal   : 83.14%
   - Stunting : 16.86%


In [102]:
# ============================================
# CELL 8: Testing Multiple Sample
# ============================================

print("\n" + "="*50)
print("        TESTING MULTIPLE SAMPLE")
print("="*50)

test_cases = [
    {"usia": 24.0, "gender": "Perempuan", "tinggi": 78.0, "berat": 9.5, "expected": "Normal"},
    {"usia": 48.0, "gender": "Laki-laki", "tinggi": 85.0, "berat": 11.0, "expected": "Stunting"},
    {"usia": 36.0, "gender": "Laki-laki", "tinggi": 95.0, "berat": 14.0, "expected": "Normal"},
    {"usia": 60.0, "gender": "Perempuan", "tinggi": 100.0, "berat": 14.5, "expected": "Normal"},
    {"usia": 30.0, "gender": "Laki-laki", "tinggi": 82.0, "berat": 10.0, "expected": "Stunting"},
]

for i, case in enumerate(test_cases, 1):
    print(f"\n--- Case {i} ---")
    print(f"Input: Usia {case['usia']} bln, {case['gender']}, Tinggi {case['tinggi']} cm, Berat {case['berat']} kg")
    print(f"Expected: {case['expected']}")

    status, conf, probs = predict_stunting(case["usia"], case["gender"], case["tinggi"], case["berat"])

    print(f"Result  : {status} (Confidence: {conf:.2f}%)")
    print(f"Status  : {'✅ CORRECT' if status == case['expected'] else '❌ WRONG'}")


        TESTING MULTIPLE SAMPLE

--- Case 1 ---
Input: Usia 24.0 bln, Perempuan, Tinggi 78.0 cm, Berat 9.5 kg
Expected: Normal

📝 Input features:
   - Usia: 24.0 bulan
   - Jenis Kelamin: Perempuan → encoded: 1
   - Tinggi: 78.0 cm
   - Berat: 9.5 kg
   - IMT: 15.61
   - Tinggi/Usia: 0.9630
Result  : Stunting (Confidence: 94.01%)
Status  : ❌ WRONG

--- Case 2 ---
Input: Usia 48.0 bln, Laki-laki, Tinggi 85.0 cm, Berat 11.0 kg
Expected: Stunting

📝 Input features:
   - Usia: 48.0 bulan
   - Jenis Kelamin: Laki-laki → encoded: 0
   - Tinggi: 85.0 cm
   - Berat: 11.0 kg
   - IMT: 15.22
   - Tinggi/Usia: 0.7265
Result  : Stunting (Confidence: 99.97%)
Status  : ✅ CORRECT

--- Case 3 ---
Input: Usia 36.0 bln, Laki-laki, Tinggi 95.0 cm, Berat 14.0 kg
Expected: Normal

📝 Input features:
   - Usia: 36.0 bulan
   - Jenis Kelamin: Laki-laki → encoded: 0
   - Tinggi: 95.0 cm
   - Berat: 14.0 kg
   - IMT: 15.51
   - Tinggi/Usia: 0.9596
Result  : Normal (Confidence: 99.95%)
Status  : ✅ CORRECT

--- 

# Cell 6: Persiapan REST API (FastAPI)
Langkah awal untuk deployment model AI sesuai milestone Minggu ke-3
.

In [103]:
# ============================================
# CELL 9: Kode API untuk Deployment (FastAPI)
# ============================================

api_code = '''
# ============================================
# app.py - REST API untuk Model SIDIAS
# ============================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Dict
import numpy as np
import joblib

# Load model dan scaler
model = joblib.load('model_sidias_rf.joblib')
scaler = joblib.load('scaler_sidias.joblib')

# Inisialisasi FastAPI
app = FastAPI(
    title="SIDIAS API",
    description="API Deteksi Stunting pada Balita Berbasis AI",
    version="1.0.0"
)

# Model Request
class BalitaInput(BaseModel):
    usia_bulan: float = Field(..., ge=0, le=120, description="Usia dalam bulan (0-120)")
    jenis_kelamin: str = Field(..., description="Laki-laki atau Perempuan")
    tinggi_badan: float = Field(..., ge=40, le=120, description="Tinggi badan dalam cm")
    berat_badan: float = Field(..., ge=2, le=30, description="Berat badan dalam kg")

# Model Response
class PredictionResponse(BaseModel):
    status: str
    confidence: float
    probabilitas: Dict[str, float]

@app.get("/")
def root():
    return {"message": "SIDIAS API is running", "version": "1.0.0"}

@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": True}

@app.post("/predict", response_model=PredictionResponse)
def predict(data: BalitaInput):
    try:
        # Validasi jenis kelamin
        if data.jenis_kelamin.lower() not in ["laki-laki", "perempuan"]:
            raise HTTPException(status_code=400, detail="Jenis kelamin harus 'Laki-laki' atau 'Perempuan'")

        # Hitung IMT
        tinggi_m = data.tinggi_badan / 100
        imt = data.berat_badan / (tinggi_m ** 2)

        # Hitung tinggi_per_usia
        tinggi_per_usia = data.tinggi_badan / (data.usia_bulan * 1.5 + 45)

        # Encode gender
        gender_encoded = 0 if data.jenis_kelamin.lower() == "laki-laki" else 1

        # Feature array
        features = np.array([[
            data.usia_bulan,
            gender_encoded,
            data.tinggi_badan,
            data.berat_badan,
            imt,
            tinggi_per_usia
        ]])

        # Scaling
        features_scaled = scaler.transform(features)

        # Prediksi
        prediction = model.predict(features_scaled)[0]
        proba = model.predict_proba(features_scaled)[0]

        status = "Stunting" if prediction == 1 else "Normal"

        return PredictionResponse(
            status=status,
            confidence=float(max(proba) * 100),
            probabilitas={"Normal": float(proba[0]), "Stunting": float(proba[1])}
        )
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

print("📋 Kode API untuk deployment (simpan sebagai 'app.py'):")
print(api_code)

print("\n" + "="*50)
print("        CARA MENJALANKAN API")
print("="*50)
print("1. Simpan kode di atas sebagai 'app.py'")
print("2. Pastikan file model dan scaler ada di folder yang sama")
print("3. Install dependencies: pip install fastapi uvicorn")
print("4. Jalankan: uvicorn app:app --reload")
print("5. Buka browser: http://localhost:8000/docs")
print("="*50)

📋 Kode API untuk deployment (simpan sebagai 'app.py'):

# ============================================
# app.py - REST API untuk Model SIDIAS
# ============================================

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Dict
import numpy as np
import joblib

# Load model dan scaler
model = joblib.load('model_sidias_rf.joblib')
scaler = joblib.load('scaler_sidias.joblib')

# Inisialisasi FastAPI
app = FastAPI(
    title="SIDIAS API",
    description="API Deteksi Stunting pada Balita Berbasis AI",
    version="1.0.0"
)

# Model Request
class BalitaInput(BaseModel):
    usia_bulan: float = Field(..., ge=0, le=120, description="Usia dalam bulan (0-120)")
    jenis_kelamin: str = Field(..., description="Laki-laki atau Perempuan")
    tinggi_badan: float = Field(..., ge=40, le=120, description="Tinggi badan dalam cm")
    berat_badan: float = Field(..., ge=2, le=30, description="Berat badan dalam kg")

# Model Response
cla